# 01 · Environment Setup

Prepares a Colab session for the crop disease diagnosis pipeline:

1. Mount Google Drive.
2. Clone the project repository into Drive (first run only) — code and notebooks.
3. Install pinned dependencies.
4. Create the data folders on Drive — datasets and checkpoints, outside the repository.
5. Verify library versions and GPU.

Run top to bottom. Safe to re-run.

| Drive folder | Holds | In Git |
|---|---|---|
| `MyDrive/Sproutly-app/` | repository: code, notebooks | yes |
| `MyDrive/CropDiseaseDiagnosis/` | datasets, models, reports, logs | no |

## 1–2 · Drive and repository

In [1]:
from pathlib import Path
import sys

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/BilliartMuthoni/confidence-calibrated-crop-diagnosis.git"
REPO_DIR = Path("/content/drive/MyDrive/Sproutly-app")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

!git -C {REPO_DIR} status --short --branch

ML_DIR = REPO_DIR / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

import config

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
## main...origin/main
 M ml/notebooks/01_environment_setup.ipynb


In [2]:
# Keep the Drive copy of the repository in sync with GitHub.
!git -C {REPO_DIR} config core.fileMode false
!git -C {REPO_DIR} pull --ff-only
!git -C {REPO_DIR} status --short --branch


Already up to date.
## main...origin/main
 M ml/notebooks/01_environment_setup.ipynb


## 3 · Dependencies

In [3]:
%pip install -q -r {ML_DIR}/requirements.txt

## 4 · Data folders on Drive

In [4]:
config.ensure_dirs()

print("Code:", config.ML_ROOT)
print("Data:", config.DATA_ROOT)
for folder in [
    config.PLANTVILLAGE_DIR, config.FIELDPLANT_DIR, config.PLANTDOC_DIR,
    config.PROCESSED_DIR, config.SPLITS_DIR, config.METADATA_DIR,
    config.MODELS_DIR, config.REPORTS_DIR, config.LOGS_DIR,
]:
    print("  ✓", folder.relative_to(config.DATA_ROOT))

Code: /content/drive/MyDrive/Sproutly-app/ml
Data: /content/drive/MyDrive/CropDiseaseDiagnosis
  ✓ datasets/raw/plantvillage
  ✓ datasets/raw/fieldplant
  ✓ datasets/raw/plantdoc
  ✓ datasets/processed
  ✓ datasets/splits
  ✓ datasets/metadata
  ✓ models
  ✓ reports
  ✓ logs


## 5 · Environment check

In [5]:
import cv2
import numpy as np
import pandas as pd
import PIL
import sklearn
import timm
import torch
import transformers

print(f"Python        {sys.version.split()[0]}")
print(f"PyTorch       {torch.__version__}")
print(f"Transformers  {transformers.__version__}")
print(f"timm          {timm.__version__}")
print(f"NumPy         {np.__version__}")
print(f"Pandas        {pd.__version__}")
print(f"Pillow        {PIL.__version__}")
print(f"OpenCV        {cv2.__version__}")
print(f"scikit-learn  {sklearn.__version__}")
print()
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    print(f"CUDA version:   {torch.version.cuda}")

Python        3.13.15
PyTorch       2.11.0+cu130
Transformers  5.18.0
timm          1.0.30
NumPy         2.1.3
Pandas        2.2.3
Pillow        11.3.0
OpenCV        4.14.0
scikit-learn  1.6.1

CUDA available: True
GPU:            Tesla T4
CUDA version:   13.0


In [6]:
from datetime import date

snapshot = config.REPORTS_DIR / f"environment_{date.today()}.txt"
!pip freeze > "{snapshot}"
print("Saved", snapshot)


Saved /content/drive/MyDrive/CropDiseaseDiagnosis/reports/environment_2026-10-10.txt


## Save to GitHub

Commits everything changed in the repository on Drive (code and notebooks only —
datasets are outside the repository) and pushes the current branch.

Needs two Colab secrets (key icon, left sidebar): `GITHUB_TOKEN` (a GitHub
personal access token with repo access) and `GIT_EMAIL`. Save the notebook
(Ctrl+S) before running this cell.

In [15]:
from google.colab import userdata

from src import github_sync

print(github_sync.status(REPO_DIR))
print()
print(github_sync.push(
    REPO_DIR,
    branch="colab/environment-setup",
    message="Run environment setup notebook",
    token=userdata.get("GITHUB_TOKEN"),
    name="BilliartMuthoni",
    email=userdata.get("GIT_EMAIL"),
))

## main...origin/main
 M ml/notebooks/01_environment_setup.ipynb



TypeError: push() got an unexpected keyword argument 'branch'